# Scenario 2: Parallel Development with Branching

---

**The Challenge:**  
DataCart has three developers that need to make **breaking schema changes simultaneously** in preparation for the Spring Sale. In a traditional single-database setup, their migrations could conflict and block each other. Furthermore, the team could perform these schema changes and testing on a smaller dataset that is not representative of production data. 

**The Lakebase Solution: Branching**  
Each developer creates an isolated **branch** — a zero-copy snapshot of production. They work independently, test their changes, and then merge back when ready. The production branch is never touched during development.

| Developer | Team | Task |
|-----------|------|------|
| Developer A | Loyalty Team | Add `loyalty_points` column + new `loyalty_members` table |
| Developer B | Global Team | Add `exchange_rates` table + convert `currency` to a FK |
| Developer C | Performance Team | Add indexes to `products` for Spring Sale traffic surge |

> 💡 **Key Insight:** With Lakebase branches, Developer A's DDL changes cannot break Developer B's code, because they work in **completely isolated copies** of the database.

### Step 0: Install Dependencies & Configure Helpers

In [0]:
%pip install databricks-sdk --upgrade -q
%pip install psycopg2-binary -q

In [0]:
dbutils.library.restartPython()

In [0]:
from databricks.sdk import WorkspaceClient
import time
import psycopg2

w = WorkspaceClient()

# Derive project name from the current user's identity
current_user = w.current_user.me()
db_user = current_user.user_name
username_prefix = db_user.split("@")[0].replace(".", "-")
project_name = f"lakebase-branching-workshop-{username_prefix}"

# List branches — the default 'production' branch should exist
branches = list(w.postgres.list_branches(parent=f"projects/{project_name}"))

print(f"📋 Branches in '{project_name}':")
for b in branches:
    branch_id = b.name.split("/branches/")[-1]
    is_default = "⭐ default" if b.status and b.status.default else ""
    print(f"   • {branch_id} {is_default}")

# Get the production branch (the default one, or fallback to the first)
prod_branch = next(
    (b for b in branches if b.status and b.status.default),
    branches[0]
)
prod_branch_name = prod_branch.name
print(f"\n✅ Production branch: {prod_branch_name}")

## Helper: Connect to Any Branch

This function is used by all scenario notebooks (01–05) to connect to a specific branch.
It handles endpoint discovery, waiting, and OAuth token generation.

In [0]:
def connect_to_branch(branch_id, wait_seconds=300):
    """
    Connect to a Lakebase branch endpoint.
    Automatically creates a compute endpoint if none exists.
    
    Args:
        branch_id: Branch name (e.g. "dev-readonly", "feature-loyalty-tier")
        wait_seconds: Max seconds to wait for endpoint to become ready (default 300)
    
    Returns:
        tuple: (connection, host, endpoint_name)
    """
    from databricks.sdk.service.postgres import Endpoint, EndpointSpec, EndpointType, Duration as Dur

    branch_full = f"projects/{project_name}/branches/{branch_id}"
    
    # Check if an endpoint already exists
    endpoints = list(w.postgres.list_endpoints(parent=branch_full))
    
    if not endpoints:
        # Create a compute endpoint for this branch
        ep_id = f"ep-{branch_id}"
        print(f"🔄 Creating compute endpoint for branch '{branch_id}'...")
        w.postgres.create_endpoint(
            parent=branch_full,
            endpoint=Endpoint(spec=EndpointSpec(
                endpoint_type=EndpointType.ENDPOINT_TYPE_READ_WRITE,
                autoscaling_limit_min_cu=min_cu,
                autoscaling_limit_max_cu=max_cu,
                suspend_timeout_duration=Dur(seconds=suspend_timeout_seconds)
            )),
            endpoint_id=ep_id
        ).wait()
        print(f"   ✅ Compute endpoint created!")
        endpoints = list(w.postgres.list_endpoints(parent=branch_full))
    
    # Wait for the endpoint host to be available
    ep = endpoints[0]
    if not ep.status or not ep.status.hosts or not ep.status.hosts.host:
        print(f"⏳ Waiting for endpoint to become ready...")
        for i in range(wait_seconds // 10):
            time.sleep(10)
            endpoints = list(w.postgres.list_endpoints(parent=branch_full))
            ep = endpoints[0]
            if ep.status and ep.status.hosts and ep.status.hosts.host:
                break
            print(f"   Still waiting... ({(i+1)*10}s)")
    
    if not ep.status or not ep.status.hosts or not ep.status.hosts.host:
        raise Exception(f"Endpoint not ready for branch '{branch_id}' after {wait_seconds}s")
    
    host = ep.status.hosts.host
    
    # Generate OAuth token and connect
    cred = w.postgres.generate_database_credential(endpoint=ep.name)
    branch_conn = psycopg2.connect(
        host=host,
        port=5432,
        dbname="databricks_postgres",
        user=db_user,
        password=cred.token,
        sslmode="require"
    )
    branch_conn.autocommit = True
    
    print(f"✅ Connected to branch '{branch_id}'")
    print(f"   Host: {host}")
    return branch_conn, host, ep.name

def delete_branch_safe(branch_id, max_retries=6, wait_between=30):
    """
    Delete a branch, retrying if the endpoint is still reconciling.
    
    Args:
        branch_id: Branch name (e.g. "dev-readonly")
        max_retries: Max number of retry attempts (default 6)
        wait_between: Seconds to wait between retries (default 30)
    """
    branch_full = f"projects/{project_name}/branches/{branch_id}"
    
    for attempt in range(max_retries):
        try:
            w.postgres.delete_branch(name=branch_full).wait()
            print(f"🗑️ Branch '{branch_id}' deleted.")
            return
        except Exception as e:
            if "reconciliation" in str(e).lower() and attempt < max_retries - 1:
                print(f"   ⏳ Endpoint still reconciling, retrying in {wait_between}s... (attempt {attempt + 1}/{max_retries})")
                time.sleep(wait_between)
            else:
                raise

def print_table(cols, rows, max_rows=30):
    if not cols:
        print("(no results)")
        return
    widths = [max(len(str(c)), max((len(str(r[i])) for r in rows), default=0)) for i, c in enumerate(cols)]
    sep    = "+" + "+".join("-" * (w + 2) for w in widths) + "+"
    print(sep)
    print("|" + "|".join(f" {c:<{widths[i]}} " for i, c in enumerate(cols)) + "|")
    print(sep)
    for row in rows[:max_rows]:
        print("|" + "|".join(f" {str(v):<{widths[i]}} " for i, v in enumerate(row)) + "|")
    print(sep)

print("🔧 print_table, connect_to_branch() and delete_branch_safe() helpers defined.")

---
## 👤 Developer A — Loyalty Team

**Goal:** Add a `loyalty_points` column to the `users` table and create a new `loyalty_members` table to support DataCart's Spring Sale loyalty program.

### Task A-1: Create Branch `dev-loyalty`

Developer A creates an isolated branch from `production`. This is a **zero-copy snapshot** — no data is duplicated on disk. The branch diverges only as changes are made.

In [0]:
from databricks.sdk.service.postgres import Branch, BranchSpec, Duration

BRANCH_NAME = "dev-loyalty"

# Fixed configuration
db_schema = "ecommerce"
min_cu = 0.5
max_cu = 4.0
suspend_timeout_seconds = 1800

# Clean up from previous runs
try:
    w.postgres.delete_branch(name=f"projects/{project_name}/branches/{BRANCH_NAME}").wait()
    print(f"🧹 Cleaned up existing branch '{BRANCH_NAME}'")
except Exception:
    pass

# Create your feature branch
print(f"\n🔄 Creating branch '{BRANCH_NAME}' from production...")
w.postgres.create_branch(
    parent=f"projects/{project_name}",
    branch=Branch(spec=BranchSpec(
        source_branch=prod_branch_name,
        ttl=Duration(seconds=172800)  # 48-hour TTL
    )),
    branch_id=BRANCH_NAME
).wait()
print(f"✅ Branch '{BRANCH_NAME}' created!")

### Task A-2: Add `loyalty_points` Column to `Customers`

Developer A runs their DDL migration on the isolated `dev-loyalty` branch. This change is **invisible to production** and to the other developers' branches.

In [0]:
# connect to dev-loyalty branch
conn_loyalty, _, _ = connect_to_branch(BRANCH_NAME)

In [0]:
print("🔧 Developer A: Adding loyalty features to 'dev-loyalty' branch...\n")

# Add loyalty_points column to users
with conn_loyalty.cursor() as cur:
    cur.execute(f"""
    ALTER TABLE {db_schema}.customers
    ADD COLUMN IF NOT EXISTS loyalty_points INT NOT NULL DEFAULT 0;
""")

# Backfill some loyalty points based on order history
with conn_loyalty.cursor() as cur:
    cur.execute(f"""
    UPDATE {db_schema}.customers u
    SET loyalty_points = (
        SELECT COALESCE(SUM(FLOOR(o.total)::INT), 0)
        FROM {db_schema}.orders o WHERE o.customer_id = u.id
    );
""") 

print("✅ Added 'loyalty_points' column and backfilled from order history.")

# Show updated users
with conn_loyalty.cursor() as cur:
    cur.execute(f"""
    SELECT id, name, loyalty_points
    FROM {db_schema}.customers
    ORDER BY loyalty_points DESC
    LIMIT 10;
""")
    cols, rows = [d[0] for d in cur.description], cur.fetchall()
print("\n🏆 Users with loyalty points (dev-loyalty branch):")
print_table(cols, rows)
conn_loyalty.close()

### Task A-3: Create the `loyalty_members` Table

Developer A also creates an entirely new table — this schema change exists **only on the `dev-loyalty` branch**.

In [0]:
# connect to dev-loyalty branch
conn_loyalty, _, _ = connect_to_branch(BRANCH_NAME)

In [0]:
# Create loyalty_members table for customers with enough points
with conn_loyalty.cursor() as cur:
    cur.execute(f"""CREATE TABLE IF NOT EXISTS {db_schema}.loyalty_members (
        id              SERIAL PRIMARY KEY,
        email           VARCHAR(255) NOT NULL REFERENCES {db_schema}.customers(email),
        tier            VARCHAR(20) NOT NULL DEFAULT 'Bronze'
            CHECK (tier IN ('Bronze', 'Silver', 'Gold', 'Platinum')),
        enrolled_at     TIMESTAMP   NOT NULL DEFAULT NOW(),
        total_earned    INT         NOT NULL DEFAULT 0
    );
""")

# Enroll customers with enough points
with conn_loyalty.cursor() as cur:
    cur.execute(f"""
    INSERT INTO {db_schema}.loyalty_members (email, tier, enrolled_at, total_earned)
    SELECT
        email,
        CASE
            WHEN loyalty_points >= 3000 THEN 'Platinum'
            WHEN loyalty_points >= 1500 THEN 'Gold'
            WHEN loyalty_points >= 500  THEN 'Silver'
            ELSE 'Bronze'
        END,
        NOW(),
        loyalty_points
    FROM {db_schema}.customers
    WHERE loyalty_points > 0
    ON CONFLICT (id) DO NOTHING;
""")

with conn_loyalty.cursor() as cur:
    cur.execute(f"""
    SELECT lm.id, u.name, lm.tier, lm.total_earned AS points
    FROM {db_schema}.loyalty_members lm
    JOIN {db_schema}.customers u ON u.email = lm.email
    ORDER BY lm.total_earned DESC
    LIMIT 10;
""")
    cols, rows = [d[0] for d in cur.description], cur.fetchall()
print("✅ Created 'loyalty_members' table and enrolled customers:")
print_table(cols, rows)
conn_loyalty.close()

### Task A-4: Verify Production Branch is UNCHANGED ✅

This is the critical test. Connect to the **`production`** branch and confirm that:
1. `users` still has **no** `loyalty_points` column

This proves that branches provide true **schema isolation**.

In [0]:
# connect to production branch
conn_prod, conn_host, conn_endpoint = connect_to_branch('production')

In [0]:
print("🔍 Checking production branch schema...\n")

# Check columns on users table in production
with conn_prod.cursor() as cur:
    cur.execute(f"""
    SELECT column_name, data_type, column_default, table_schema, table_name
    FROM information_schema.columns
    WHERE table_schema = '{db_schema}' AND table_name = 'customers'
    ORDER BY ordinal_position;
""")
    prod_columns = [row[0] for row in cur.fetchall()]

print(f"📋 Production branch columns: {prod_columns}")
print(f"   Has loyalty_points? {'loyalty_points' in prod_columns}")
print("\n" + "=" * 60)
print("🎯 RESULT: 'loyalty_points' and 'loyalty_members' exist ONLY")
print("   in 'dev-loyalty'. Production schema is untouched!")
print("=" * 60)

Perform checks to validate `loyalty_members` table does **not exist** in Production

---
## 🌍 Developer B — Global Team

**Goal:** Refactor the `orders` table to support true multi-currency by replacing the `currency` varchar column with a foreign key to a new `exchange_rates` table.

This is a **breaking schema change** — it would have caused a conflict with Developer A's work if they shared a database. With Lakebase branching, it's completely isolated.

### Task B-1: Create Branch `modify-orders`

In [0]:
from databricks.sdk.service.postgres import Branch, BranchSpec, Duration

BRANCH_NAMEV2 = "modify-orders"

# Clean up from previous runs
try:
    w.postgres.delete_branch(name=f"projects/{project_name}/branches/{BRANCH_NAMEV2}").wait()
    print(f"🧹 Cleaned up existing branch '{BRANCH_NAMEV2}'")
except Exception:
    pass

# Create your feature branch
print(f"\n🔄 Creating branch '{BRANCH_NAMEV2}' from production...")
w.postgres.create_branch(
    parent=f"projects/{project_name}",
    branch=Branch(spec=BranchSpec(
        source_branch=prod_branch_name,
        ttl=Duration(seconds=172800)  # 48-hour TTL
    )),
    branch_id=BRANCH_NAMEV2
).wait()
print(f"✅ Branch '{BRANCH_NAMEV2}' created!")

### Task B-2: Create the `exchange_rates` Table

Developer B first creates the reference table that the new foreign key will point to.

In [0]:
# connect to orders branch
conn_orders, conn_host, conn_endpoint = connect_to_branch('modify-orders')

In [0]:
print("🔧 Developer B: Building multi-currency support in 'modify-orders' branch...\n")

# create exchange_rates table
with conn_orders.cursor() as cur:
    cur.execute(f"""
    CREATE TABLE IF NOT EXISTS {db_schema}.exchange_rates (
        id              SERIAL PRIMARY KEY,
        currency_code   CHAR(3)         UNIQUE NOT NULL,
        currency_name   VARCHAR(100)    NOT NULL,
        rate_to_usd     NUMERIC(12, 6)  NOT NULL,
        last_updated    TIMESTAMP       NOT NULL DEFAULT NOW()
    );
""")

# insert data into the exchange_rates table    
with conn_orders.cursor() as cur:
    cur.execute(f"""
    INSERT INTO {db_schema}.exchange_rates (currency_code, currency_name, rate_to_usd) VALUES
        ('USD', 'US Dollar',          1.000000),
        ('EUR', 'Euro',               1.085000),
        ('GBP', 'British Pound',      1.265000),
        ('JPY', 'Japanese Yen',       0.006700),
        ('AED', 'UAE Dirham',         0.272300),
        ('INR', 'Indian Rupee',       0.012000),
        ('BRL', 'Brazilian Real',     0.200000),
        ('CNY', 'Chinese Yuan',       0.138000)
    ON CONFLICT (currency_code) DO NOTHING;
""")

with conn_orders.cursor() as cur:
    cur.execute(f"""
    SELECT currency_code, currency_name, rate_to_usd
    FROM {db_schema}.exchange_rates ORDER BY currency_code;
""")
    cols, rows = [d[0] for d in cur.description], cur.fetchall()
print("✅ Created 'exchange_rates' table with live rates:")
print_table(cols, rows)

### Task B-3: Migrate `orders.currency` to a Foreign Key

Developer B adds a new FK column `currency_id`, migrates data from the old `currency` varchar, and can then drop the old column. This migration is entirely contained within the `modify-orders` branch.

In [0]:
print("🔧 Migrating orders.currency varchar → FK to exchange_rates...\n")

# Step 1: Add new FK column
with conn_orders.cursor() as cur:
    cur.execute(f"""
    ALTER TABLE {db_schema}.orders
    ADD COLUMN IF NOT EXISTS currency_id INT REFERENCES {db_schema}.exchange_rates(id);
""")

# Step 2: Populate FK from existing currency codes
with conn_orders.cursor() as cur:
    cur.execute(f"""
    UPDATE {db_schema}.orders o
    SET currency_id = er.id
    FROM {db_schema}.exchange_rates er
    WHERE er.currency_code = o.currency;
""")

# Step 3: Make the FK NOT NULL (all rows have been migrated)
with conn_orders.cursor() as cur:
    cur.execute(f"""
        ALTER TABLE {db_schema}.orders ALTER COLUMN currency_id SET NOT NULL;
""")

# Step 4: Drop the old varchar column
with conn_orders.cursor() as cur:
    cur.execute(f"""
        ALTER TABLE {db_schema}.orders DROP COLUMN currency;
""") 
    
print("✅ Migration complete. Lets verifying the result...\n")

In [0]:
print("Verifying the result...\n")

with conn_orders.cursor() as cur:
    cur.execute(f"""
    SELECT o.id, u.name AS customer, p.name AS product,
           o.quantity, o.total,
           er.currency_code, er.rate_to_usd,
           ROUND(o.total * er.rate_to_usd, 2) AS total_usd
    FROM {db_schema}.orders o
    JOIN {db_schema}.customers         u  ON u.id  = o.customer_id
    JOIN {db_schema}.products      p  ON p.id  = o.product_id
    JOIN {db_schema}.exchange_rates er ON er.id = o.currency_id
    ORDER BY o.id
    LIMIT 10;
""")
    cols, rows = [d[0] for d in cur.description], cur.fetchall()
print("📊 Orders with normalised currency FK (modify-orders branch):")
print_table(cols, rows)

# Confirm production still has varchar currency column
conn_prod, conn_host, conn_endpoint = connect_to_branch('production')
with conn_prod.cursor() as cur:
    cur.execute(f"""
    SELECT column_name, data_type
    FROM information_schema.columns
    WHERE table_schema = '{db_schema}' AND table_name = 'orders'
    ORDER BY ordinal_position;
""")
    cols2, rows2 = [d[0] for d in cur.description], cur.fetchall()
print("\n📋 'currency' column in orders table in PRODUCTION (still has varchar currency):")
print_table(cols2, rows2)

conn_orders.close()
conn_prod.close()

---
## ⚡ Developer C — Performance Team

**Goal:** Create indexes on the `products` table to handle the high-traffic surge expected during the Spring Sale. Under load, full table scans on `products` would be catastrophic.

### Task C-1: Create Branch `add-index`

In [0]:
BRANCH_NAMEV3 = "add-index"

# Fixed configuration
db_schema = "ecommerce"
min_cu = 0.5
max_cu = 4.0
suspend_timeout_seconds = 1800

# Clean up from previous runs
try:
    w.postgres.delete_branch(name=f"projects/{project_name}/branches/{BRANCH_NAMEV3}").wait()
    print(f"🧹 Cleaned up existing branch '{BRANCH_NAMEV3}'")
except Exception:
    pass

# Create your feature branch
print(f"\n🔄 Creating branch '{BRANCH_NAMEV3}' from production...")
w.postgres.create_branch(
    parent=f"projects/{project_name}",
    branch=Branch(spec=BranchSpec(
        source_branch=prod_branch_name,
        ttl=Duration(seconds=172800)  # 48-hour TTL
    )),
    branch_id=BRANCH_NAMEV3
).wait()
print(f"✅ Branch '{BRANCH_NAMEV3}' created!")

In [0]:
conn_index, conn_host, conn_endpoint = connect_to_branch('add-index')

### Task C-2: Create Performance Indexes on `products`

Developer C creates three indexes on the `add-index` branch:
- **`idx_products_category`** — speeds up category browsing (most common query pattern)
- **`idx_products_price`** — speeds up price-range filter queries  
- **`idx_products_stock_qty`** — speeds up "in stock" filter queries

These can be validated on the branch before promoting to production.

In [0]:
print("🔧 Developer C: Creating performance indexes on 'add-index' branch...\n")

index_statements = [
    ("idx_products_price",
     f"CREATE INDEX IF NOT EXISTS idx_products_price ON {db_schema}.products (price);",
     "Price-range filter queries")
]

for name, sql, purpose in index_statements:
    with conn_index.cursor() as cur:
        cur.execute(sql)
    print(f"   ✅ Created: {name} ({purpose})")

# Verify indexes exist
with conn_index.cursor() as cur:
    cur.execute(f"""
    SELECT indexname, indexdef
    FROM pg_indexes
    WHERE schemaname = '{db_schema}' AND tablename = 'products'
    ORDER BY indexname;
""")
    cols, rows = [d[0] for d in cur.description], cur.fetchall()
print("\n📋 Indexes on 'products' in add-index branch:")
print_table(cols, rows)

# Confirm production has NO extra indexes yet
conn_prod, conn_host, conn_endpoint = connect_to_branch('production')
with conn_prod.cursor() as cur:
    cur.execute(f"""
    SELECT indexname, indexdef
    FROM pg_indexes
    WHERE schemaname = 'public' AND tablename = 'products'
    ORDER BY indexname;
""")
    cols2, rows2 = [d[0] for d in cur.description], cur.fetchall()
print("\n📋 Indexes on 'products' in PRODUCTION branch (no custom indexes yet):")
print_table(cols2, rows2)

conn_index.close()
conn_prod.close()

print("\n" + "=" * 60)
print("🎯 SUMMARY: Three developers worked in PARALLEL with zero conflicts.")
print("   Each branch has isolated changes ready for review & merge.")
print("=" * 60)

## 🎯 Summary

Each developer created an isolated **branch** to accomplish their tasks in an isolated environment. They work independently, test their changes, and then merge back when ready. The production branch is never touched during development.

| Developer | Team | Task |
|-----------|------|------|
| Developer A | Loyalty Team | Add `loyalty_points` column + new `loyalty_members` table |
| Developer B | Global Team | Add `exchange_rates` table + convert `currency` to a FK |
| Developer C | Performance Team | Add indexes to `products` for Spring Sale traffic surge |